In [8]:
############################################################
# STEP3.2_01_distance_both.R
#  - Compute correlation-based distance (1 - r)
#    (1) WITH n_eff correction  → "signlessCorr"
#    (2) WITHOUT n_eff correction → "signlessCorr_noNeff"
#
#  - Outputs:
#    for_checking_20251127/sub/01_distance_STEP3.2_signlessCorr/run_<run_id>/
#      variables/{OH,FP}/distance_corr_variables_<dataset>_<run_id>.csv
#      samples/{A...D}/distance_corr_samples_<dataset>_<run_id>.csv
#
#    for_checking_20251127/sub/01_distance_STEP3.2_signlessCorr_noNeff/run_<run_id>/
#      variables/{OH,FP}/distance_corrNoNeff_variables_<dataset>_<run_id>.csv
#      samples/{A...D}/distance_corrNoNeff_samples_<dataset>_<run_id>.csv
#
# 使い方:
#   Rscript STEP3.2_01_distance_both.R
#   Rscript STEP3.2_01_distance_both.R 20251128_123045
############################################################

## ---- Settings ----
root_base <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"
input_dir <- file.path(root_base, "data", "for_MDS_STEP2")

target_vars <- c("PCEmax","Jsc","Voc","FF")

datasets_variables <- list(
  OH = list(file = "preprocessed_features_OH.csv"),
  FP = list(file = "preprocessed_features_FP.csv")
)

datasets_samples <- list(
  A_OH_plus_others = list(file = "preprocessed_features_OH.csv"),
  B_FP_plus_others = list(file = "preprocessed_features_FP.csv"),
  C_OH_only        = list(file = "preprocessed_features_OH_only.csv"),
  D_FP_only        = list(file = "preprocessed_features_FP_only.csv")
)

###############################################
# run_id の決定（Jupyter では override を優先）
###############################################

# ← この override_run_id を自分で決める
override_run_id <- "20251130_153500"   # ★ ここだけ変えれば全部そろう

# 引数（Rscript時のみ使う）
args <- commandArgs(trailingOnly = TRUE)

if (!is.na(override_run_id)) {
  run_id <- override_run_id
} else if (length(args) >= 1 && grepl("^\\d{8}_\\d{6}$", args[1])) {
  run_id <- args[1]
} else {
  run_id <- format(Sys.time(), "%Y%m%d_%H%M%S")
}

cat(">>> STEP3.2_01_distance_both — run_id:", run_id, "\n")

# 出力先ベース
step01_base_neff    <- file.path(root_base, "sub", "01_distance_STEP3.2_signlessCorr")
step01_base_noneff  <- file.path(root_base, "sub", "01_distance_STEP3.2_signlessCorr_noNeff")

step01_run_neff   <- file.path(step01_base_neff,   paste0("run_", run_id))
step01_run_noneff <- file.path(step01_base_noneff, paste0("run_", run_id))

dir.create(step01_run_neff,   showWarnings = FALSE, recursive = TRUE)
dir.create(step01_run_noneff, showWarnings = FALSE, recursive = TRUE)

dir.create(file.path(step01_run_neff,   "variables"), showWarnings = FALSE, recursive = TRUE)
dir.create(file.path(step01_run_neff,   "samples"),   showWarnings = FALSE, recursive = TRUE)
dir.create(file.path(step01_run_noneff, "variables"), showWarnings = FALSE, recursive = TRUE)
dir.create(file.path(step01_run_noneff, "samples"),   showWarnings = FALSE, recursive = TRUE)

## ---- Helpers ----
read_numeric_explanatory <- function(path, target_vars) {
  if (!file.exists(path)) {
    stop("[ERROR] Input file not found: ", path)
  }
  df <- read.delim(path, header = TRUE, sep = ",", row.names = 1,
                   as.is = TRUE, strip.white = FALSE)
  expl_vars <- df[, !(colnames(df) %in% target_vars), drop = FALSE]
  is_char <- sapply(expl_vars, function(col) {
    any(grepl("[A-Za-z]", col) & col != "NA", na.rm = TRUE)
  })
  numData <- expl_vars[, !is_char, drop = FALSE]
  if (!ncol(numData)) {
    stop("[ERROR] No numeric explanatory columns detected in: ", path)
  }
  return(numData)
}

# ==== (1) n_eff 補正あり距離 ====
compute_corr_distance_neff <- function(numData, unit = c("variables","samples")) {
  unit <- match.arg(unit)
  X <- as.matrix(numData)

  if (unit == "variables") {
    N <- nrow(X)
    corM <- suppressWarnings(cor(X, use = "pairwise.complete.obs"))
    mask <- !is.na(X)
    mask_num <- matrix(as.numeric(mask), nrow = N)
    n_eff <- t(mask_num) %*% mask_num  # p x p
    dimnames(n_eff) <- list(colnames(X), colnames(X))

    d_prime <- 1 - corM
    scale_mat <- matrix(N, nrow = ncol(X), ncol = ncol(X)) / n_eff
    d_corr <- d_prime * scale_mat

    bad <- (n_eff <= 1) | is.na(d_corr)
    d_corr[bad] <- 2
  } else {
    n_samples <- nrow(X)
    M <- ncol(X)
    corM <- suppressWarnings(cor(t(X), use = "pairwise.complete.obs"))
    mask <- !is.na(X)
    mask_num <- matrix(as.numeric(mask), nrow = n_samples)
    n_eff <- mask_num %*% t(mask_num)  # n x n
    rownames(n_eff) <- rownames(X)
    colnames(n_eff) <- rownames(X)

    d_prime <- 1 - corM
    scale_mat <- matrix(M, nrow = n_samples, ncol = n_samples) / n_eff
    d_corr <- d_prime * scale_mat

    bad <- (n_eff <= 1) | is.na(d_corr)
    d_corr[bad] <- 2
  }

  diag(d_corr) <- 0
  d_corr <- pmin(pmax(d_corr, 0), 2)
  d_corr <- (d_corr + t(d_corr)) / 2
  list(dist_matrix = d_corr)
}

# ==== (2) n_eff 補正なし距離 (純 1 - r) ====
compute_corr_distance_noneff <- function(numData, unit = c("variables","samples")) {
  unit <- match.arg(unit)
  X <- as.matrix(numData)

  if (unit == "variables") {
    corM <- suppressWarnings(cor(X, use = "pairwise.complete.obs"))
  } else {
    corM <- suppressWarnings(cor(t(X), use = "pairwise.complete.obs"))
  }

  d_corr <- 1 - corM
  bad <- is.na(d_corr)
  d_corr[bad] <- 2
  diag(d_corr) <- 0
  d_corr <- pmin(pmax(d_corr, 0), 2)
  d_corr <- (d_corr + t(d_corr)) / 2
  list(dist_matrix = d_corr)
}

process_one_dataset_distance <- function(unit, dataset_id, file_name) {
  in_path <- file.path(input_dir, file_name)
  cat("  [", unit, "] ", dataset_id, " → ", in_path, "\n", sep = "")

  numData <- read_numeric_explanatory(in_path, target_vars)

  # (1) n_eff 補正あり
  dist_neff   <- compute_corr_distance_neff(numData, unit = unit)$dist_matrix
  # (2) n_eff 補正なし
  dist_noneff <- compute_corr_distance_noneff(numData, unit = unit)$dist_matrix

  # ---- 出力用フォルダ ----
  out_dir_neff   <- file.path(step01_run_neff,   unit, dataset_id)
  out_dir_noneff <- file.path(step01_run_noneff, unit, dataset_id)
  dir.create(out_dir_neff,   showWarnings = FALSE, recursive = TRUE)
  dir.create(out_dir_noneff, showWarnings = FALSE, recursive = TRUE)

  # ---- (1) n_eff版 ----
  out_file_neff <- file.path(
    out_dir_neff,
    sprintf("distance_corr_%s_%s_%s.csv", unit, dataset_id, run_id)
  )
  write.csv(dist_neff, out_file_neff, row.names = TRUE)
  cat("    [SAVED n_eff]     ", out_file_neff, "\n", sep = "")

  # ---- (2) no n_eff版 ----
  out_file_noneff <- file.path(
    out_dir_noneff,
    sprintf("distance_corrNoNeff_%s_%s_%s.csv", unit, dataset_id, run_id)
  )
  write.csv(dist_noneff, out_file_noneff, row.names = TRUE)
  cat("    [SAVED no_n_eff]  ", out_file_noneff, "\n", sep = "")
}

## ---- Execute ----
cat("\n--- Variables (OH/FP) ---\n")
for (ds in names(datasets_variables)) {
  process_one_dataset_distance("variables", ds, datasets_variables[[ds]]$file)
}

cat("\n--- Samples (A–D) ---\n")
for (ds in names(datasets_samples)) {
  process_one_dataset_distance("samples", ds, datasets_samples[[ds]]$file)
}

cat("\n✅ STEP3.2_01_distance_both finished. run_id =", run_id, "\n")
cat("   Output roots:\n")
cat("   - with n_eff   :", step01_run_neff, "\n")
cat("   - without n_eff:", step01_run_noneff, "\n")


>>> STEP3.2_01_distance_both <U+2014> run_id: 20251130_153500 

--- Variables (OH/FP) ---
  [variables] OH <U+2192> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/for_MDS_STEP2/preprocessed_features_OH.csv
    [SAVED n_eff]     /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/01_distance_STEP3.2_signlessCorr/run_20251130_153500/variables/OH/distance_corr_variables_OH_20251130_153500.csv
    [SAVED no_n_eff]  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/01_distance_STEP3.2_signlessCorr_noNeff/run_20251130_153500/variables/OH/distance_corrNoNeff_variables_OH_20251130_153500.csv
  [variables] FP <U+2192> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/for_MD

In [4]:
# ############################################################
# # STEP3.2_01_distance.R
# #  - Compute correlation-based distance with n_eff correction
# #  - Outputs:
# #    for_checking_20251127/01_distance_STEP3.2_signlessCorr/run_<run_id>/
# #       variables/{OH,FP}/distance_corr_variables_<dataset>_<run_id>.csv
# #       samples/{A...D}/distance_corr_samples_<dataset>_<run_id>.csv
# #
# # 使い方:
# #   Rscript STEP3.2_01_distance.R                # run_id 自動
# #   Rscript STEP3.2_01_distance.R 20251128_123045  # run_id 明示指定
# ############################################################

# ## ---- Settings ----
# root_base <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"
# input_dir <- file.path(root_base, "data", "for_MDS_STEP2")

# target_vars <- c("PCEmax","Jsc","Voc","FF")

# datasets_variables <- list(
#   OH = list(file = "preprocessed_features_OH.csv"),
#   FP = list(file = "preprocessed_features_FP.csv")
# )

# datasets_samples <- list(
#   A_OH_plus_others = list(file = "preprocessed_features_OH.csv"),
#   B_FP_plus_others = list(file = "preprocessed_features_FP.csv"),
#   C_OH_only        = list(file = "preprocessed_features_OH_only.csv"),
#   D_FP_only        = list(file = "preprocessed_features_FP_only.csv")
# )

# ###############################################
# # run_id の決定（Jupyter では override を優先）
# ###############################################

# # ← この override_run_id を自分で決める
# override_run_id <- "20251130_153500"   # ★ ここだけ変えれば全部そろう

# # 引数（Rscript時のみ使う）
# args <- commandArgs(trailingOnly = TRUE)

# if (!is.na(override_run_id)) {
#   # ★ override が有効ならそれを使う
#   run_id <- override_run_id

# } else if (length(args) >= 1 && grepl("^\\d{8}_\\d{6}$", args[1])) {
#   # ターミナルからの実行用
#   run_id <- args[1]

# } else {
#   # どちらも無い場合は timestamp
#   run_id <- format(Sys.time(), "%Y%m%d_%H%M%S")
# }

# cat(">>> STEP3.2_01_distance — run_id:", run_id, "\n")


# # 出力先を sub フォルダ直下に変更
# step01_base <- file.path(root_base, "sub", "01_distance_STEP3.2_signlessCorr")
# step01_run_dir <- file.path(step01_base, paste0("run_", run_id))
# dir.create(step01_run_dir, showWarnings = FALSE, recursive = TRUE)
# dir.create(file.path(step01_run_dir, "variables"), showWarnings = FALSE, recursive = TRUE)
# dir.create(file.path(step01_run_dir, "samples"),   showWarnings = FALSE, recursive = TRUE)

# ## ---- Helpers ----
# read_numeric_explanatory <- function(path, target_vars) {
#   if (!file.exists(path)) {
#     stop("[ERROR] Input file not found: ", path)
#   }
#   df <- read.delim(path, header = TRUE, sep = ",", row.names = 1,
#                    as.is = TRUE, strip.white = FALSE)
#   expl_vars <- df[, !(colnames(df) %in% target_vars), drop = FALSE]
#   is_char <- sapply(expl_vars, function(col) {
#     any(grepl("[A-Za-z]", col) & col != "NA", na.rm = TRUE)
#   })
#   numData <- expl_vars[, !is_char, drop = FALSE]
#   if (!ncol(numData)) {
#     stop("[ERROR] No numeric explanatory columns detected in: ", path)
#   }
#   return(numData)
# }

# # 距離計算
# compute_corr_distance <- function(numData, unit = c("variables","samples")) {
#   unit <- match.arg(unit)
#   X <- as.matrix(numData)

#   if (unit == "variables") {
#     N <- nrow(X)
#     corM <- suppressWarnings(cor(X, use = "pairwise.complete.obs"))
#     mask <- !is.na(X)
#     mask_num <- matrix(as.numeric(mask), nrow = N)
#     n_eff <- t(mask_num) %*% mask_num  # p x p
#     dimnames(n_eff) <- list(colnames(X), colnames(X))
#     d_prime <- 1 - corM
#     scale_mat <- matrix(N, nrow = ncol(X), ncol = ncol(X)) / n_eff
#     d_corr <- d_prime * scale_mat
#     bad <- (n_eff <= 1) | is.na(d_corr)
#     d_corr[bad] <- 2
#     diag(d_corr) <- 0
#     d_corr <- pmin(pmax(d_corr, 0), 2)
#     d_corr <- (d_corr + t(d_corr)) / 2
#     list(dist_matrix = d_corr)
#   } else {
#     n_samples <- nrow(X)
#     M <- ncol(X)
#     corM <- suppressWarnings(cor(t(X), use = "pairwise.complete.obs"))
#     mask <- !is.na(X)
#     mask_num <- matrix(as.numeric(mask), nrow = n_samples)
#     n_eff <- mask_num %*% t(mask_num)  # n x n
#     rownames(n_eff) <- rownames(X)
#     colnames(n_eff) <- rownames(X)
#     d_prime <- 1 - corM
#     scale_mat <- matrix(M, nrow = n_samples, ncol = n_samples) / n_eff
#     d_corr <- d_prime * scale_mat
#     bad <- (n_eff <= 1) | is.na(d_corr)
#     d_corr[bad] <- 2
#     diag(d_corr) <- 0
#     d_corr <- pmin(pmax(d_corr, 0), 2)
#     d_corr <- (d_corr + t(d_corr)) / 2
#     list(dist_matrix = d_corr)
#   }
# }

# process_one_dataset_distance <- function(unit, dataset_id, file_name) {
#   in_path <- file.path(input_dir, file_name)
#   cat("  [", unit, "] ", dataset_id, " → ", in_path, "\n", sep = "")

#   numData <- read_numeric_explanatory(in_path, target_vars)
#   dist_res <- compute_corr_distance(numData, unit = unit)
#   d_mat <- dist_res$dist_matrix

#   out_dir_unit <- file.path(step01_run_dir, unit, dataset_id)
#   dir.create(out_dir_unit, showWarnings = FALSE, recursive = TRUE)

#   out_file <- file.path(
#     out_dir_unit,
#     sprintf("distance_corr_%s_%s_%s.csv", unit, dataset_id, run_id)
#   )
#   write.csv(d_mat, out_file, row.names = TRUE)
#   cat("    [SAVED] ", out_file, "\n", sep = "")
# }

# ## ---- Execute ----
# cat("\n--- Variables (OH/FP) ---\n")
# for (ds in names(datasets_variables)) {
#   process_one_dataset_distance("variables", ds, datasets_variables[[ds]]$file)
# }

# cat("\n--- Samples (A–D) ---\n")
# for (ds in names(datasets_samples)) {
#   process_one_dataset_distance("samples", ds, datasets_samples[[ds]]$file)
# }

# cat("\n✅ STEP3.2_01_distance finished. run_id =", run_id, "\n")
# cat("   Output root:", step01_run_dir, "\n")


>>> STEP3.2_01_distance <U+2014> run_id: 20251130_153500 

--- Variables (OH/FP) ---
  [variables] OH <U+2192> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/for_MDS_STEP2/preprocessed_features_OH.csv
    [SAVED] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/01_distance_STEP3.2_signlessCorr/run_20251130_153500/variables/OH/distance_corr_variables_OH_20251130_153500.csv
  [variables] FP <U+2192> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/for_MDS_STEP2/preprocessed_features_FP.csv
    [SAVED] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/01_distance_STEP3.2_signlessCorr/run_20251130_153500/variables/FP/distance_corr_variables_FP_20251130_153500.csv



[INFO] File A: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr/run_20251130_153500/variables/FP/MDS_linear_top3_FP_20251130_153500.csv 
[INFO] File B: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/02_mds_STEP3.2_signlessCorr/run_20251130_153500/variables/FP/MDScoords_linear_top3_variables_FP_20251130_153500.csv 

===== Comparison Result =====
[1] TRUE


In [7]:
# ############################################################
# # STEP3.2_01_distance_noNeff.R
# #  - Compute correlation-based distance WITHOUT n_eff correction
# #    (pure 1 - r distance with pairwise.complete.obs)
# #
# #  - Outputs:
# #    for_checking_20251127/sub/01_distance_STEP3.2_signlessCorr_noNeff/run_<run_id>/
# #       variables/{OH,FP}/distance_corrNoNeff_variables_<dataset>_<run_id>.csv
# #       samples/{A...D}/distance_corrNoNeff_samples_<dataset>_<run_id>.csv
# #
# # 使い方:
# #   Rscript STEP3.2_01_distance_noNeff.R                # run_id 自動
# #   Rscript STEP3.2_01_distance_noNeff.R 20251128_123045  # run_id 明示指定
# ############################################################

# ## ---- Settings ----
# root_base <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"
# input_dir <- file.path(root_base, "data", "for_MDS_STEP2")

# target_vars <- c("PCEmax","Jsc","Voc","FF")

# datasets_variables <- list(
#   OH = list(file = "preprocessed_features_OH.csv"),
#   FP = list(file = "preprocessed_features_FP.csv")
# )

# datasets_samples <- list(
#   A_OH_plus_others = list(file = "preprocessed_features_OH.csv"),
#   B_FP_plus_others = list(file = "preprocessed_features_FP.csv"),
#   C_OH_only        = list(file = "preprocessed_features_OH_only.csv"),
#   D_FP_only        = list(file = "preprocessed_features_FP_only.csv")
# )

# ###############################################
# # run_id の決定（Jupyter では override を優先）
# ###############################################

# # ← この override_run_id を自分で決める
# override_run_id <- "20251130_153500"   # ★ ここだけ変えれば全部そろう

# # 引数（Rscript時のみ使う）
# args <- commandArgs(trailingOnly = TRUE)

# if (!is.na(override_run_id)) {
#   # ★ override が有効ならそれを使う
#   run_id <- override_run_id

# } else if (length(args) >= 1 && grepl("^\\d{8}_\\d{6}$", args[1])) {
#   # ターミナルからの実行用
#   run_id <- args[1]

# } else {
#   # どちらも無い場合は timestamp
#   run_id <- format(Sys.time(), "%Y%m%d_%H%M%S")
# }

# cat(">>> STEP3.2_01_distance_noNeff — run_id:", run_id, "\n")


# # 出力先: n_eff補正あり版と区別するためフォルダ名を変更
# step01_base <- file.path(root_base, "sub", "01_distance_STEP3.2_signlessCorr_noNeff")
# step01_run_dir <- file.path(step01_base, paste0("run_", run_id))
# dir.create(step01_run_dir, showWarnings = FALSE, recursive = TRUE)
# dir.create(file.path(step01_run_dir, "variables"), showWarnings = FALSE, recursive = TRUE)
# dir.create(file.path(step01_run_dir, "samples"),   showWarnings = FALSE, recursive = TRUE)

# ## ---- Helpers ----
# read_numeric_explanatory <- function(path, target_vars) {
#   if (!file.exists(path)) {
#     stop("[ERROR] Input file not found: ", path)
#   }
#   df <- read.delim(path, header = TRUE, sep = ",", row.names = 1,
#                    as.is = TRUE, strip.white = FALSE)
#   expl_vars <- df[, !(colnames(df) %in% target_vars), drop = FALSE]
#   is_char <- sapply(expl_vars, function(col) {
#     any(grepl("[A-Za-z]", col) & col != "NA", na.rm = TRUE)
#   })
#   numData <- expl_vars[, !is_char, drop = FALSE]
#   if (!ncol(numData)) {
#     stop("[ERROR] No numeric explanatory columns detected in: ", path)
#   }
#   return(numData)
# }

# # 距離計算（n_eff補正なし版）
# compute_corr_distance_noNeff <- function(numData, unit = c("variables","samples")) {
#   unit <- match.arg(unit)
#   X <- as.matrix(numData)

#   if (unit == "variables") {
#     # 変数間の相関
#     corM <- suppressWarnings(cor(X, use = "pairwise.complete.obs"))
#   } else {
#     # サンプル間の相関
#     corM <- suppressWarnings(cor(t(X), use = "pairwise.complete.obs"))
#   }

#   # 基本距離: d = 1 - r
#   d_corr <- 1 - corM

#   # 相関が計算できなかったペアは最大距離 (=2) に設定
#   bad <- is.na(d_corr)
#   d_corr[bad] <- 2

#   # 対角成分は 0
#   diag(d_corr) <- 0

#   # 0〜2 の範囲にクリップ
#   d_corr <- pmin(pmax(d_corr, 0), 2)

#   # 浮動小数の誤差などで非対称になる可能性があるので対称化
#   d_corr <- (d_corr + t(d_corr)) / 2

#   list(dist_matrix = d_corr)
# }

# process_one_dataset_distance <- function(unit, dataset_id, file_name) {
#   in_path <- file.path(input_dir, file_name)
#   cat("  [", unit, "] ", dataset_id, " → ", in_path, "\n", sep = "")

#   numData <- read_numeric_explanatory(in_path, target_vars)
#   dist_res <- compute_corr_distance_noNeff(numData, unit = unit)
#   d_mat <- dist_res$dist_matrix

#   out_dir_unit <- file.path(step01_run_dir, unit, dataset_id)
#   dir.create(out_dir_unit, showWarnings = FALSE, recursive = TRUE)

#   # ★ ファイル名も n_eff なし版と分かるように変更
#   out_file <- file.path(
#     out_dir_unit,
#     sprintf("distance_corrNoNeff_%s_%s_%s.csv", unit, dataset_id, run_id)
#   )
#   write.csv(d_mat, out_file, row.names = TRUE)
#   cat("    [SAVED] ", out_file, "\n", sep = "")
# }

# ## ---- Execute ----
# cat("\n--- Variables (OH/FP) [no n_eff correction] ---\n")
# for (ds in names(datasets_variables)) {
#   process_one_dataset_distance("variables", ds, datasets_variables[[ds]]$file)
# }

# cat("\n--- Samples (A–D) [no n_eff correction] ---\n")
# for (ds in names(datasets_samples)) {
#   process_one_dataset_distance("samples", ds, datasets_samples[[ds]]$file)
# }

# cat("\n✅ STEP3.2_01_distance_noNeff finished. run_id =", run_id, "\n")
# cat("   Output root:", step01_run_dir, "\n")


>>> STEP3.2_01_distance_noNeff <U+2014> run_id: 20251130_153500 

--- Variables (OH/FP) [no n_eff correction] ---
  [variables] OH <U+2192> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/for_MDS_STEP2/preprocessed_features_OH.csv
    [SAVED] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/01_distance_STEP3.2_signlessCorr_noNeff/run_20251130_153500/variables/OH/distance_corrNoNeff_variables_OH_20251130_153500.csv
  [variables] FP <U+2192> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/for_MDS_STEP2/preprocessed_features_FP.csv
    [SAVED] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/01_distance_STEP3.2_signlessCorr_noNeff/run_20251130_153500/variables/FP